# FaultFalcon - fine-tune SmolLM2 to talk like an on-call assistant (free T4, ~10 min)

Uses ONLY the synthetic conversation dataset (no project code): on the laptop run `make chat-data`, which writes

- `data/training/conversations_train.jsonl` (dev incidents, 3 dialogue styles each)
- `data/training/conversations_eval.jsonl` (held-out incidents, for the validation loss only)

**Runtime -> Change runtime type -> T4 GPU**, then run the cells in order. The method is LoRA (r=16 on every
attention and MLP projection), with the loss on the assistant turns only; the adapter is merged into the base
weights at the end, so the result is a plain SmolLM2 checkpoint the app loads with `FF_MODEL_PATH`.

In [ ]:
# 1) upload the two dataset files (select both in the file dialog; skipped when they are already here)
from google.colab import files
import json, os
if not all(os.path.exists(f) for f in ("conversations_train.jsonl", "conversations_eval.jsonl")):
    files.upload()
train = [json.loads(l) for l in open("conversations_train.jsonl", encoding="utf-8")]
evals = [json.loads(l) for l in open("conversations_eval.jsonl", encoding="utf-8")]
print(len(train), "training dialogues,", len(evals), "evaluation dialogues")
print(json.dumps(train[0]["messages"][:3], indent=1)[:1500])

In [ ]:
# 2) libraries (Colab already has torch; peft adds LoRA). Colab's preinstalled torchao is too old for recent
#    peft ("incompatible version of torchao") and is not needed here, so it is removed.
#    If you already hit that error: run this cell, then Runtime -> Restart session, then run from cell 1 again.
!pip -q uninstall -y torchao
!pip -q install "transformers>=4.44,<5" "peft>=0.12" accelerate
import torch
print(torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch to T4!)")

In [ ]:
# 3) settings
BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"   # the app's default model (FF_MODEL=smollm2-360m)
OUT = "smollm2-360m-ff-chat"                   # folder name of the fine-tuned model
EPOCHS, LR, GRAD_ACCUM, MAX_LEN = 3, 2e-4, 8, 2048
LORA_R, LORA_ALPHA = 16, 32

In [ ]:
# 4) tokenize: the same ChatML format the app uses; labels only on assistant turns (+ their <|im_end|>)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE)

def encode(messages):
    ids, labels = [], []
    for m in messages:
        head = tok(f"<|im_start|>{m['role']}\n", add_special_tokens=False).input_ids
        body = tok(m["content"] + "<|im_end|>\n", add_special_tokens=False).input_ids
        ids += head + body
        labels += [-100] * len(head) + (body if m["role"] == "assistant" else [-100] * len(body))
    return ids[:MAX_LEN], labels[:MAX_LEN]

train_ids = [encode(d["messages"]) for d in train]
eval_ids = [encode(d["messages"]) for d in evals]
print("longest dialogue:", max(len(x) for x, _ in train_ids), "tokens;",
      sum(sum(l != -100 for l in y) for _, y in train_ids), "trained tokens")

In [ ]:
# 5) LoRA training (float32 weights, fp16 autocast on the T4)
import math, random, time
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).cuda()
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = math.ceil(len(train_ids) * EPOCHS / GRAD_ACCUM)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, steps // 10)) * (1 - s / steps))
scaler = torch.cuda.amp.GradScaler()

def loss_of(ids, labels):
    x = torch.tensor([ids]).cuda()
    y = torch.tensor([labels]).cuda()
    with torch.autocast("cuda", dtype=torch.float16):
        return model(input_ids=x, labels=y).loss

@torch.no_grad()
def eval_loss():
    model.eval()
    out = sum(float(loss_of(i, l)) for i, l in eval_ids) / len(eval_ids)
    model.train()
    return out

print(f"validation loss before: {eval_loss():.3f}")
rng, t0, step = random.Random(0), time.time(), 0
model.train()
for epoch in range(EPOCHS):
    order = train_ids[:]
    rng.shuffle(order)
    for k, (ids, labels) in enumerate(order, 1):
        scaler.scale(loss_of(ids, labels) / GRAD_ACCUM).backward()
        if k % GRAD_ACCUM == 0 or k == len(order):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
    print(f"epoch {epoch + 1}: validation loss {eval_loss():.3f}  ({(time.time() - t0) / 60:.1f} min)")

In [ ]:
# 6) try it on a held-out dialogue: the first engineer message, the model's answer vs the dataset's answer
d = evals[0]["messages"]
prompt = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in d[:2]) + "<|im_start|>assistant\n"
model = model.cuda().eval()  # cell 7 moves the model to the CPU for saving: bring it back (safe to re-run)
x = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(next(model.parameters()).device)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    y = model.generate(x, max_new_tokens=120, do_sample=False, eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"))
print("ENGINEER:", d[1]["content"][-300:])
print("\nFINE-TUNED:", tok.decode(y[0, x.shape[1]:], skip_special_tokens=True))
print("\nDATASET:", d[2]["content"])

In [ ]:
# 7) merge LoRA into the weights and save (float16 safetensors, ~0.7 GB). The tokenizer files are copied from the
#    base model as published, so the app's older transformers 4.33 / tokenizers 0.13 read them unchanged.
import shutil
from huggingface_hub import hf_hub_download
merged = model.merge_and_unload().to(torch.float16).cpu()
shutil.rmtree(OUT, ignore_errors=True)
merged.save_pretrained(OUT, safe_serialization=True)
for f in ("tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt", "special_tokens_map.json"):
    try:
        shutil.copy(hf_hub_download(BASE, f), f"{OUT}/{f}")
    except Exception as e:
        print("skip", f, e)
print(sorted(os.listdir(OUT)))

In [ ]:
# 8) download (zip); if the browser download stalls, run the next cell instead
!zip -qr {OUT}.zip {OUT}
!ls -lh {OUT}.zip
files.download(f"{OUT}.zip")

In [ ]:
# optional: copy to Google Drive and download it from drive.google.com
from google.colab import drive
drive.mount("/content/drive")
!cp {OUT}.zip /content/drive/MyDrive/

**On the laptop:** unzip the file inside the project so it becomes `models/smollm2-360m-ff-chat/`, then

    make chat-ft FT=models/smollm2-360m-ff-chat INC=INC-109     # the guided chat with the fine-tuned model
    set FF_MODEL_PATH=models/smollm2-360m-ff-chat && make app   # (PowerShell: $env:FF_MODEL_PATH="models/...")

`make finetune-eval FT=models/smollm2-360m-ff-chat` compares it with the base model on the held-out incidents.